# H2 corrective arm — amplitude-only frequency loss (~10 h, one session)

**Question this answers:** H2 predicts that phase gradient consistency improves restoration
*over amplitude-only supervision*. No arm in the thesis isolates that comparison: the baseline
has no frequency loss at all, and `abl_phase` adds both halves of `PhaseEnhancedFrequencyLoss`
together. This notebook trains the missing comparator.

**The arm.** `abl_freqamp` is `abl_phase` with the phase half switched off — same outer weight
(0.20), same amplitude half (0.5), `freq_phase_weight = 0.0`. Everything else is identical to
Set 1: 200 epochs, seed 42, 256 px, batch 4/GPU × 2, LR 2.8e-4 cosine, EMA, edge + enhance base.
One variable differs, so the two subtractions below are clean:

| contrast | what it isolates |
|---|---|
| `abl_freqamp − abl_baseline` | the amplitude half alone |
| `abl_phase − abl_freqamp` | **the phase half, on top of amplitude — exactly H2's comparator** |

**Optional second arm.** `abl_freqphs` (amplitude half off, phase half on) is defined but
commented out. It answers whether phase works *without* amplitude, which H2 does not ask.
Enable it only if you want the full 2×2; it costs another ~10 h.

## Decision rule — fixed before any result exists

Primary test: clip-level Wilcoxon on 30 clip means + cluster-bootstrap CI (`clustered_analysis.py`,
20,000 resamples), on **`abl_phase − abl_freqamp`**, all four metrics, final checkpoint.

| outcome | reading for the thesis |
|---|---|
| phase-half contrast significant on ≥ 3 of 4 metrics | **H2 supported as stated.** The phase half adds over amplitude-only supervision. |
| significant on 1–2 metrics | H2 partially supported; name the metrics. |
| significant on none | H2 not supported. The frequency term's gain is not attributable to phase; §5.2.2 and the SC-1 hypothesis are revised accordingly. |

Read PSNR against the per-difference run-to-run SD (√2 × 0.1358 ≈ 0.19 dB), not 0.1358.
`abl_freqamp − abl_baseline` is reported alongside, descriptively, in the same table.
Report whatever comes back, in both directions. A single seed; say so.

⚠️ **This is a reinterpretation arm, not a re-run.** It does not change any existing number.
It adds the comparator H2 named and the thesis never trained.

**Inputs:** `physir-code` (with the `freq_amp_weight` / `freq_phase_weight` keys — hard-guarded),
`lolblur-dataset`. **GPU T4 × 2.** Continuation sessions: attach this notebook's previous output.

**Outputs** (drop straight into the existing analysis folders):
`per_image/per_image_abl_freqamp_{last,ep150,best}.csv` → `.metrics/per_image_corrected/`
`per_image_dists/per_image_dists_abl_freqamp_{...}.csv` → `.metrics/per_image_dists/`

## 1. Install (torch-swap guard)

In [ ]:
import torch as _t0
_TORCH_BEFORE, _CUDA_BEFORE = _t0.__version__, _t0.cuda.is_available()
!pip install -q lpips pytorch-msssim einops ptflops pyiqa

import importlib.util as _iu, subprocess, sys
_need = ['pyiqa', 'lpips', 'pytorch_msssim', 'einops', 'ptflops', 'yaml', 'pandas']
_missing = [m for m in _need if _iu.find_spec(m) is None]
assert not _missing, f'install failed for {_missing} — re-run this cell (Internet must be ON)'
_after = subprocess.run([sys.executable, '-c',
    'import torch; print(torch.__version__, torch.cuda.is_available())'],
    capture_output=True, text=True).stdout.split()
assert _after[0] == _TORCH_BEFORE and _after[1] == str(_CUDA_BEFORE), (
    f'pip changed torch {_TORCH_BEFORE}/{_CUDA_BEFORE} -> {_after} — restart kernel and pin torch')
print(f'torch {_TORCH_BEFORE}, cuda {_CUDA_BEFORE} — unchanged ✅ | deps present ✅')

## 2. Paths + stale-code guards

In [ ]:
import os, sys, glob, re
CODE_DIR     = '/kaggle/input/datasets/jeffreyboahen/physir-code'
LOLBLUR_ROOT = '/kaggle/input/datasets/jeffreyboahen/lolblur-dataset'
WORKING_DIR  = '/kaggle/working'
def find_root(base, marker, depth=3):
    if os.path.exists(os.path.join(base, marker)): return base
    for d in range(1, depth+1):
        h = sorted(glob.glob(os.path.join(base, *(['*']*d), marker)))
        if h: return h[0][:-len(marker)].rstrip(os.sep)
    return None
CODE_ROOT = find_root(CODE_DIR, 'train.py'); assert CODE_ROOT, f'physir-code not found under {CODE_DIR}'
if CODE_ROOT not in sys.path: sys.path.insert(0, CODE_ROOT)
src = open(os.path.join(CODE_ROOT,'train.py')).read()
miss = [n for n in ('restore_phys_state','class ModelEMA','best_psnr',
                    'snapshot_epochs','time_budget_h','Seeded RNGs','_atomic_save') if n not in src]
assert not miss, f'STALE physir-code, missing {miss} — re-upload with the current train.py + losses/.'
LOLBLUR = find_root(LOLBLUR_ROOT, os.path.join('test','low_blur_noise')) or LOLBLUR_ROOT
assert os.path.isdir(os.path.join(LOLBLUR, 'train')) and os.path.isdir(os.path.join(LOLBLUR, 'test')), LOLBLUR
print('CODE_ROOT:', CODE_ROOT); print('LOLBLUR  :', LOLBLUR)

# ── HARD GUARD 1: the loss factory must expose the two half-weights ──────────
# Without them freq_phase_weight would be silently ignored and the arm would train as a
# duplicate of abl_phase — 10 h that answer nothing.
_li = open(os.path.join(CODE_ROOT,'losses','__init__.py')).read()
assert 'freq_amp_weight' in _li and 'freq_phase_weight' in _li, (
    'STALE physir-code: losses/__init__.py does not pass freq_amp_weight / freq_phase_weight. '
    'Re-upload physir-code with the updated losses/__init__.py.')

# ── HARD GUARD 2: the test loader must not shuffle (per-image pairing) ───────
_ds = open(os.path.join(CODE_ROOT,'data','dataset_reader','dataset_LOLBlur.py')).read()
_bad = [d for d in re.findall(r'DataLoader\([^)]*\)', _ds, re.S) if 'shuffle=True' in d.replace(' ','')]
assert not _bad, 'STALE physir-code: test DataLoader still has shuffle=True.'

# ── HARD GUARD 3: runtime proof, not a string match ──────────────────────────
import torch
from losses import create_physics_loss
from losses.loss import PhaseEnhancedFrequencyLoss
_cfg = {'retinex': False, 'blur': False, 'phase': True, 'phase_weight': 0.20}
_full, _ = create_physics_loss(_cfg, 'cpu')
_amp,  _ = create_physics_loss({**_cfg, 'freq_phase_weight': 0.0}, 'cpu')
_m = _amp['phase_loss']
assert (_m.weight, _m.amp_weight, _m.phase_weight) == (0.20, 0.5, 0.0), vars(_m)
torch.manual_seed(0); _p, _t = torch.rand(1,3,64,64), torch.rand(1,3,64,64)
assert torch.equal(_full['phase_loss'](_p,_t), PhaseEnhancedFrequencyLoss(weight=0.20)(_p,_t)), \
    'default config no longer reproduces the original loss — do not run'
print('✅ guards passed — half-weights wired, default bit-identical, loader unshuffled')

## 2b. Session restore (attach this notebook's previous output as an input)

In [ ]:
import shutil
ARMS = ('abl_freqamp', 'abl_freqphs')     # exact prefixes — never restores other arms' files
def _restore():
    n = 0
    for pat in ('/kaggle/input/*/{a}*.pt', '/kaggle/input/*/*/{a}*.pt', '/kaggle/input/*/*/*/{a}*.pt'):
        for a in ARMS:
            for src in glob.glob(pat.format(a=a)):
                b = os.path.basename(src)
                if not re.fullmatch(rf'{a}(_s\d+)?_(latest|best)(_ep\d+)?\.pt', b): continue
                dst = os.path.join(WORKING_DIR, b)
                if not os.path.exists(dst): shutil.copy2(src, dst); n += 1
    for base in ('metrics_out', '.metrics'):
        for depth in ('*', '*/*', '*/*/*'):
            for src in glob.glob(f'/kaggle/input/{depth}/{base}/*/*.csv'):
                run = os.path.basename(os.path.dirname(src))
                if not run.startswith(ARMS): continue
                dstd = os.path.join(WORKING_DIR, '.metrics', run); os.makedirs(dstd, exist_ok=True)
                dst = os.path.join(dstd, os.path.basename(src))
                if not os.path.exists(dst): shutil.copy2(src, dst); n += 1
    print(f'restored {n} files from previous session' if n else 'fresh session — nothing to restore')
_restore()

## 3. Train/val split (90/10, seed 42 — same as all stages)

In [ ]:
import random
from pathlib import Path
SUBS=['low_blur_noise','high_sharp_scaled']
TRAIN_SRC=Path(LOLBLUR)/'train'; SPLIT=Path(WORKING_DIR)/'LOLBlur_split'
random.seed(42)   # split seed is FIXED across all stages/seeds — never change
clips=sorted([d.name for d in (TRAIN_SRC/'low_blur_noise').iterdir() if d.is_dir()])
n_val=max(1,int(len(clips)*0.1)); val=set(random.sample(clips,n_val)); tr=[c for c in clips if c not in val]
for nm,cl in [('physir_train',tr),('physir_val',val)]:
    for sub in SUBS:
        b=SPLIT/nm/sub; b.mkdir(parents=True,exist_ok=True)
        for c in cl:
            d=b/c
            if not d.exists(): d.symlink_to((TRAIN_SRC/sub/c).resolve())
PHYSIR_TRAIN=str(SPLIT/'physir_train'); PHYSIR_VAL=str(SPLIT/'physir_val')
print('train',len(tr),'val',len(val))

## 4. Config — matched to Set 1 `abl_phase` in everything but `freq_phase_weight`

In [ ]:
import yaml
EPOCHS   = 200      # UNIFORM ablation budget — matched to abl_phase
SNAPSHOT = 150      # frozen mid-run checkpoint, as every suite arm
SEED     = 42       # suite seed; split seed stays 42 regardless
RUN_TAG  = '' if SEED == 42 else f'_s{SEED}'
BUDGET_H = 0        # 0 = DISABLED: train until the session ends; atomic 2-epoch checkpoints resume

# phase = OUTER weight of PhaseEnhancedFrequencyLoss (abl_phase used 0.20);
# amp / phs = its two inner half-weights (the loss's own default is 0.5 / 0.5).
CONFIGS = {
  'abl_freqamp': dict(edge=True, phase=0.20, amp=0.5, phs=0.0, retinex=0.0, blur=0.0, percep=False),
# 'abl_freqphs': dict(edge=True, phase=0.20, amp=0.0, phs=0.5, retinex=0.0, blur=0.0, percep=False),
}
CONFIGS = {k + RUN_TAG: v for k, v in CONFIGS.items()}

def build_cfg(name, sp, budget_h=0):
    phys_on = (sp['phase']>0 or sp['retinex']>0 or sp['blur']>0)
    cfg = {
      'seed': SEED,
      'datasets': {'name':'LOLBlur',
        'train': {'low_path':f'{PHYSIR_TRAIN}/low_blur_noise','high_path':f'{PHYSIR_TRAIN}/high_sharp_scaled',
                  'crop_size':256,'batch_size':4,'n_workers':4,'use_flips':True},
        'val': {'test_path':PHYSIR_VAL,'batch_size_test':1,'verbose':False,'n_workers':4}},
      'network': {'name':'DarkIR','img_channels':3,'width':32,'middle_blk_num_enc':2,'middle_blk_num_dec':2,
                  'enc_blk_nums':[1,2,3],'dec_blk_nums':[3,1,1],'dilations':[1,4,9],'extra_depth_wise':True},
      'save': {'pretrained':None,'path':f'{WORKING_DIR}/{name}_latest.pt','best':f'{WORKING_DIR}/{name}_best.pt',
               'save_freq':2,'eval_freq':10,'perceptual_eval_epochs':[],
               'snapshot_epochs':[SNAPSHOT]},
      'pixel_criterion':'l1',
      'perceptual': sp['percep'], 'perceptual_weight':0.01,
      'perceptual_type':'lpips','perceptual_net':'vgg',
      'perceptual_criterion':'l1','perceptual_reduction':'mean',
      'edge': sp['edge'],'edge_weight':0.05,'edge_criterion':'l2','edge_reduction':'mean',
      'frequency':False,
      'enhance':True,'enhance_weight':0.01,'enhance_criterion':'l1','enhance_reduction':'mean',
      'physics_losses': {'enabled':phys_on,'phase2_start':9999,'phase3_start':9999,'phase4_start':1,
                         'retinex': sp['retinex']>0,'retinex_weight':sp['retinex'],
                         'blur': sp['blur']>0,'blur_weight':sp['blur'],
                         'phase': sp['phase']>0,'phase_weight':sp['phase'],
                         'freq_amp_weight': sp['amp'], 'freq_phase_weight': sp['phs']},
      'train': {'epochs':EPOCHS,'lr_initial':0.00028,'weight_decay':0.001,'lr_scheme':'CosineAnnealing',
                'eta_min':1e-6,'grad_clip':1.0,'time_budget_h':budget_h,
                'ema':{'enabled':True,'decay':0.9999,'warmup':2000}},
      'wandb':{'init':False,'name':name},'Resize':False,
    }
    p=f'{WORKING_DIR}/{name}.yml'
    yaml.dump(cfg, open(p,'w'), default_flow_style=False)
    return p, cfg

# Pre-flight: build each config and instantiate its loss from the YAML that train.py will read.
for name, sp in CONFIGS.items():
    p, _ = build_cfg(name, sp)
    pl, _ = create_physics_loss(yaml.safe_load(open(p))['physics_losses'], 'cpu')
    m = pl['phase_loss']
    assert (m.weight, m.amp_weight, m.phase_weight) == (sp['phase'], sp['amp'], sp['phs']), (name, vars(m))
    print(f'{name}: L_freq = {m.weight} × ({m.amp_weight}·amp + {m.phase_weight}·phase) ✅')
print('EPOCHS', EPOCHS, '| snapshot @', SNAPSHOT, '| seed', SEED)

## 5. TRAIN loop — idempotent, resume-safe, never self-terminates

In [ ]:
import csv, subprocess

def done_ep(name):
    ck = os.path.join(WORKING_DIR, f'{name}_latest.pt')
    if os.path.isfile(ck):
        try: return int(torch.load(ck, map_location='cpu', weights_only=False).get('epoch', 0))
        except Exception as e: print(f'  ({name}_latest.pt unreadable: {e} — falling back to log)')
    lg = Path(WORKING_DIR)/'.metrics'/name/'training_log.csv'
    if not lg.exists(): return 0
    last = 0
    try:
        for r in csv.DictReader(open(lg)): last = max(last, int(r['epoch']))
    except Exception: pass
    return last

def mirror_metrics():
    s = os.path.join(WORKING_DIR, '.metrics'); d = os.path.join(WORKING_DIR, 'metrics_out')
    if os.path.isdir(s): shutil.copytree(s, d, dirs_exist_ok=True)

for name, sp in CONFIGS.items():
    d = done_ep(name)
    if d >= EPOCHS: print(f'✅ {name}: trained {d}/{EPOCHS} ep — skip'); continue
    p, _ = build_cfg(name, sp, budget_h=BUDGET_H)
    print(f"\n===== TRAIN {name} {'(resume ep %d)' % d if d else '(scratch)'} → {EPOCHS} ep =====")
    r = subprocess.run(['torchrun','--nproc_per_node=2', f'{CODE_ROOT}/train.py', '-p', p])
    print(f'===== {name} exit {r.returncode} | now {done_ep(name)}/{EPOCHS} ep =====')
    mirror_metrics()
mirror_metrics()
print('\nTRAIN loop done —', {n: f'{done_ep(n)}/{EPOCHS}' for n in CONFIGS})

## 6. EVAL — LOLBlur test × {last, ep150, best}: PSNR/SSIM/LPIPS + DISTS, with `fname`

In [ ]:
import time, json, numpy as np
import train as T
from data import create_test_data
from pytorch_msssim import ssim as _ssim
import lpips as _lpips_mod, pyiqa

BATCH, dev = 4, 'cuda'
opt_test = {'name':'LOLBlur','val':{'test_path':os.path.join(LOLBLUR,'test'),'batch_size_test':BATCH},
            'train':{'n_workers':2,'verbose':False}}
raw,_ = create_test_data(rank=0, world_size=1, opt=opt_test)
LOADER = raw if not isinstance(raw, dict) else list(raw.values())[0]['loader']
DS = LOADER.dataset
# clip-qualified key (LOLBlur basenames repeat across its 30 clips) — same as the DISTS files
FNAMES = [os.path.join(os.path.basename(os.path.dirname(p)), os.path.basename(p)) for p in DS.imgs_low]
assert len(FNAMES) == len(DS) == 1800 and len(set(FNAMES)) == 1800, 'unexpected test-set keys'

LPIPS_NET = _lpips_mod.LPIPS(net='vgg', verbose=False).to(dev).eval()
DISTS = pyiqa.create_metric('dists', device=dev, as_loss=False)
_x = torch.rand(2,3,128,128, device=dev); _y = torch.rand_like(_x)
assert DISTS(_x,_y).flatten().numel() == 2, 'DISTS must return one score per image'
try: DISTS(_x*2-1, _y*2-1); _guard = False
except AssertionError: _guard = True
assert _guard, 'pyiqa [0,1] range guard is OFF — refusing to score'
print('DISTS self-check ✅ (per-sample, [0,1] guard active) | pyiqa', pyiqa.__version__)

NET = {'name':'DarkIR','img_channels':3,'width':32,'middle_blk_num_enc':2,'middle_blk_num_dec':2,
       'enc_blk_nums':[1,2,3],'dec_blk_nums':[3,1,1],'dilations':[1,4,9],'extra_depth_wise':True}
OUT_M = Path(WORKING_DIR)/'per_image';       OUT_M.mkdir(exist_ok=True)
OUT_D = Path(WORKING_DIR)/'per_image_dists'; OUT_D.mkdir(exist_ok=True)

@torch.no_grad()
def eval_model(model):
    rows, i0 = [], 0
    for high, low in LOADER:
        high, low = high.to(dev), low.to(dev)
        pred = model(low, side_loss=False).clamp(0,1)[:, :, :high.shape[2], :high.shape[3]]
        assert float(high.min()) >= 0 and float(high.max()) <= 1
        mse  = ((pred-high)**2).mean(dim=(1,2,3)); psnr = 20*torch.log10(1.0/(mse.sqrt()+1e-8))
        ss   = _ssim(pred, high, data_range=1.0, size_average=False)
        lp   = LPIPS_NET(pred*2-1, high*2-1).flatten()      # thesis convention [-1,1]
        lpdk = LPIPS_NET(pred, high).flatten()              # DarkIR's unscaled convention
        ds   = DISTS(pred, high).flatten()                  # [0,1]
        for j in range(high.shape[0]):
            rows.append((i0+j, FNAMES[i0+j], float(psnr[j]), float(ss[j]), float(lp[j]),
                         float(lpdk[j]), float(ds[j])))
        i0 += high.shape[0]
    return rows

def _write(path, header, rows):          # tmp + rename: a cut session never leaves a half CSV
    tmp = str(path) + '.tmp'
    with open(tmp, 'w', newline='') as f:
        w = csv.writer(f); w.writerow(header); w.writerows(rows)
    os.replace(tmp, path)

def ckpts(name):
    return [('last',  f'{WORKING_DIR}/{name}_latest.pt'),
            ('ep150', f'{WORKING_DIR}/{name}_latest_ep{SNAPSHOT}.pt'),
            ('best',  f'{WORKING_DIR}/{name}_best.pt')]

for name in CONFIGS:
    if done_ep(name) < EPOCHS: print(f'⏳ {name}: {done_ep(name)}/{EPOCHS} ep — finish training first'); continue
    for tag, ck_path in ckpts(name):
        pm, pd_ = OUT_M/f'per_image_{name}_{tag}.csv', OUT_D/f'per_image_dists_{name}_{tag}.csv'
        if pm.exists() and pd_.exists(): print(f'✅ {name}[{tag}] exists — skip'); continue
        if not os.path.isfile(ck_path): print(f'⚠️ {name}[{tag}]: {os.path.basename(ck_path)} missing'); continue
        ck = torch.load(ck_path, map_location='cpu', weights_only=False)
        model = T.build_model(NET).to(dev)
        model.load_state_dict({k.replace('module.',''):v for k,v in (ck.get('ema') or ck['model']).items()},
                              strict=False); model.eval()
        t0 = time.time(); rows = eval_model(model)
        _write(pm, ['idx','fname','psnr','ssim','lpips','lpips_darkirconv'], [r[:6] for r in rows])
        _write(pd_, ['idx','fname','dists'], [(r[0], r[1], r[6]) for r in rows])
        a = np.array([r[2:] for r in rows])
        print(f'{name}[{tag:5s}@ep{ck.get("best_epoch") if tag=="best" else ck.get("epoch")}] '
              f'PSNR {a[:,0].mean():.4f} SSIM {a[:,1].mean():.4f} LPIPS {a[:,2].mean():.4f} '
              f'DISTS {a[:,4].mean():.4f}  ({(time.time()-t0)/60:.1f} min)')
        del model; torch.cuda.empty_cache()
json.dump({'metric':'DISTS','impl':f'pyiqa {pyiqa.__version__}','input_range':'[0,1]',
           'backbone':'VGG16 (shared with LPIPS-vgg)','resizing':'none','check_input_range':True},
          open(OUT_D/'dists_provenance.json','w'), indent=1)
mirror_metrics(); print('\nEVAL complete.')

## 7. Leaderboard against Table 4.8, and packaging

Means only — significance is decided offline with `clustered_analysis.py` (commands printed
below). The reference rows are the archived Set 1 values at the final checkpoint.

In [ ]:
import pandas as pd
REF = {'abl_baseline': (25.2651, 0.8395, 0.2747, 0.1405),   # Table 4.8 / per_image_dists
       'abl_phase':    (26.2287, 0.8637, 0.2378, 0.1159)}
try:
    for name in CONFIGS:
        pm, pd_ = OUT_M/f'per_image_{name}_last.csv', OUT_D/f'per_image_dists_{name}_last.csv'
        if not (pm.exists() and pd_.exists()): print(f'{name}: no final-checkpoint results yet'); continue
        m, d = pd.read_csv(pm), pd.read_csv(pd_)
        assert list(m['fname']) == list(d['fname']), 'metric and DISTS files disagree on row order'
        me = (m['psnr'].mean(), m['ssim'].mean(), m['lpips'].mean(), d['dists'].mean())
        tab = pd.DataFrame({**{k: v for k, v in REF.items()}, name: me},
                           index=['PSNR ↑','SSIM ↑','LPIPS ↓','DISTS ↓']).T
        print(tab.round(4).to_string())
        b, ph = REF['abl_baseline'], REF['abl_phase']
        print(f'\n{name} − abl_baseline : PSNR {me[0]-b[0]:+.3f} dB | SSIM {me[1]-b[1]:+.4f} | '
              f'LPIPS {me[2]-b[2]:+.4f} | DISTS {me[3]-b[3]:+.4f}')
        print(f'abl_phase − {name}    : PSNR {ph[0]-me[0]:+.3f} dB | SSIM {ph[1]-me[1]:+.4f} | '
              f'LPIPS {ph[2]-me[2]:+.4f} | DISTS {ph[3]-me[3]:+.4f}   ← H2 comparator')
        print('\n(Means only. The verdict is the clip-level test below, per the decision rule in the header.)')
except Exception as e:
    print(f'⚠ leaderboard skipped ({type(e).__name__}: {e}) — CSVs are still saved')

# Packaging must never raise: the CSVs are already in the output either way.
try:
    import zipfile
    zp = os.path.join(WORKING_DIR, 'h2_freqamp_results.zip')
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as zf:
        for folder in (OUT_M, OUT_D, Path(WORKING_DIR)/'metrics_out'):
            for f in sorted(Path(folder).rglob('*')):
                if f.is_file() and f.suffix in ('.csv', '.json'):
                    zf.write(f, f.relative_to(WORKING_DIR))
    print(f'\n✅ archived → {zp} ({os.path.getsize(zp)/2**20:.1f} MB)')
except Exception as e:
    print(f'\n⚠ zip failed ({type(e).__name__}: {e}) — download per_image/ and per_image_dists/ directly')

print("""
Offline, after copying the CSVs into .metrics/per_image_corrected and .metrics/per_image_dists:
  python clustered_analysis.py .metrics/per_image_corrected --a abl_phase_last --b abl_freqamp_last \\
      --label "phase-half (H2): abl_phase - abl_freqamp" --boots 20000 --dists .metrics/per_image_dists
  python clustered_analysis.py .metrics/per_image_corrected --a abl_freqamp_last --b abl_baseline_last \\
      --label "amplitude-half: abl_freqamp - baseline" --boots 20000 --dists .metrics/per_image_dists
""")